# Lab 03. 표본분포, 표준오차와 부트스트랩

표본통계량은 표본에 따라 달라진다. 표본평균의 표준오차는 큰 표본에서 대략 다음과 같다.

$$SE(\bar{x})=\frac{s}{\sqrt{n}}$$

부트스트랩은 관측 표본에서 복원추출을 반복해 통계량의 표본분포를 근사한다.

In [ ]:
from pathlib import Path
import sys, subprocess

REPO_URL = "https://github.com/niko2204/bigdataservice.git"
if "google.colab" in sys.modules:
    ROOT = Path("/content/bigdataservice")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(ROOT)], check=True)
else:
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    ROOT = next((p.resolve() for p in candidates if (p / "src").exists()), None)
    if ROOT is None:
        raise FileNotFoundError("bigdataservice 저장소 루트에서 Notebook을 실행하세요.")

sys.path.insert(0, str(ROOT))
STUDENT_ID = "20260001"  # 반드시 본인 학번으로 변경
print("저장소:", ROOT)
print("실습 학번:", STUDENT_ID)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.education.personalized_data import make_student_dataset, student_seed

df = make_student_dataset(STUDENT_ID).drop_duplicates()
sales = df["월매출"].dropna().to_numpy()
rng = np.random.default_rng(student_seed(STUDENT_ID))
print("관측 수:", len(sales), "평균:", sales.mean())

## 1. 완성 예제: 표본 크기와 표준오차

현재 데이터를 모집단처럼 두고 서로 다른 크기의 표본을 1,000번 추출한다.

In [ ]:
records = []
for n in [10, 20, 50, 100]:
    sample_means = [
        rng.choice(sales, size=n, replace=True).mean()
        for _ in range(1000)
    ]
    records.append({
        "n": n,
        "평균의평균": np.mean(sample_means),
        "시뮬레이션_SE": np.std(sample_means, ddof=1),
        "공식_SE": np.std(sales, ddof=1) / np.sqrt(n),
    })
se_table = pd.DataFrame(records)
display(se_table.round(3))

표본 크기가 4배가 될 때 표준오차가 약 절반이 되는지 수치로 확인한다. 시뮬레이션과 공식은 반복 횟수와 분포 때문에 완전히 같지는 않다.

## 2. 완성 예제: 평균의 부트스트랩 신뢰구간

In [ ]:
bootstrap_means = np.array([
    rng.choice(sales, size=len(sales), replace=True).mean()
    for _ in range(3000)
])
ci_low, ci_high = np.percentile(bootstrap_means, [2.5, 97.5])
print(f"월매출 평균={sales.mean():.1f}, 95% bootstrap CI=({ci_low:.1f}, {ci_high:.1f})")

plt.figure(figsize=(8, 4))
plt.hist(bootstrap_means, bins=35, edgecolor="white")
plt.axvline(ci_low, color="red", linestyle="--")
plt.axvline(ci_high, color="red", linestyle="--")
plt.xlabel("부트스트랩 표본평균")
plt.ylabel("빈도")
plt.title("월매출 평균의 부트스트랩 분포")
plt.show()

## 3. 신뢰구간 해석

“모평균이 이 특정 구간에 있을 확률이 95%”라고 해석하지 않는다. 같은 절차로 표본추출과 구간 계산을 반복하면 장기적으로 약 95%의 구간이 모평균을 포함한다는 뜻이다.

## 4. 따라하기: 업종별 평균 신뢰구간

각 업종에서 별도로 부트스트랩을 수행한다. 업종별 표본 수가 다르면 구간 폭도 달라질 수 있다.

In [ ]:
rows = []
for category, group in df.groupby("업종"):
    values = group["월매출"].dropna().to_numpy()
    means = [
        rng.choice(values, size=len(values), replace=True).mean()
        for _ in range(2000)
    ]
    low, high = np.percentile(means, [2.5, 97.5])
    rows.append({"업종": category, "n": len(values), "평균": values.mean(), "하한": low, "상한": high})
display(pd.DataFrame(rows).round(1))

## 5. 독립 연습

1. 평균 대신 중앙값의 95% 부트스트랩 신뢰구간을 계산한다.
2. 반복 횟수 200, 1,000, 5,000에서 구간 끝값이 얼마나 변하는지 비교한다.
3. 이상값 포함·제외 시 평균과 중앙값 구간을 비교한다.
4. 부트스트랩으로 해결할 수 없는 표본 편향의 예를 상권 데이터에서 제시한다.

In [ ]:
# TODO: 반복 횟수별 중앙값 신뢰구간 표
median_ci_table = None
display(median_ci_table)

## 6. 자가점검

- [ ] 복원추출과 비복원추출의 차이를 설명한다.
- [ ] n과 표준오차의 제곱근 관계를 확인했다.
- [ ] 신뢰구간을 확률 문장으로 잘못 해석하지 않는다.
- [ ] 부트스트랩이 대표성 편향을 고치지 못함을 설명한다.